# Practice Lab: RAG Service with Embedding model



You will build the same pipeline — parse data, embed, store, retrieve — but with two real changes:
1. **Real data from a file**: instead of a hardcoded `sample_chunks` list, you'll read and parse `data/rag_knowledge_base.txt`, a 25-chunk RAG-topic knowledge base (more than the lab's 10 `sample_chunks`).
2. **A real embedding model, called through OpenRouter**: instead of `np.random.randn(...)` plus a keyword-triggered signal boost, you'll call OpenRouter's `/embeddings` endpoint to embed both documents and queries with a real model (`openai/text-embedding-3-small`).


## Setup

Install ChromaDB, `requests` (for calling OpenRouter), and `python-dotenv` to load your API key. Then load `OPENROUTER_API_KEY` the same way the RAG evaluation lab did.

In [ ]:
!uv pip install chromadb requests python-dotenv numpy -q


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
assert OPENROUTER_API_KEY, (
    "OPENROUTER_API_KEY not found. Add it to a .env file in this folder "
    "(same key you used for the RAG evaluation lab's judge model)."
)
print("OpenRouter API key loaded.")


## Exercise 1: Read and Parse the Knowledge Base File

`data/rag_knowledge_base.txt` contains 25 chunk entries, each separated by a line containing only `---`, formatted like this:

```
TITLE: HNSW Indexing
SECTION: algorithm
HNSW, or Hierarchical Navigable Small World, is an approximate nearest neighbor algorithm...
```

Implement `parse_knowledge_base(filepath)`:
1. Read the whole file.
2. Split it on lines containing only `---`, dropping any empty blocks.
3. For each block: the first line is `TITLE: <title>`, the second is `SECTION: <section>`, and everything after that (joined back into one string, with each line stripped) is the chunk's `text`.
4. Return a list of dicts shaped exactly like the main lab's `sample_chunks`: `{"text": ..., "metadata": {"title": ..., "section": ..., "chunk_id": 0}}`.

In [ ]:
def parse_knowledge_base(filepath: str) -> list:
    # TODO: read the file, split on '---', and parse each block's TITLE/SECTION/text
    # into a chunk dict shaped like sample_chunks in the main lab.
    pass

chunks = parse_knowledge_base("data/rag_knowledge_base.txt")
print(f"Parsed {len(chunks)} chunks")
print(chunks[0])
print(chunks[7])  # should be the HNSW Indexing chunk

from tests import rag_service_checks
rag_service_checks.check_exercise_1(chunks)

## Exercise 2: Embed the Chunks via OpenRouter

OpenRouter's `/embeddings` endpoint is OpenAI-compatible: `POST https://openrouter.ai/api/v1/embeddings` with a JSON body of `{"model": ..., "input": ...}`, where `input` can be a single string **or a list of strings** (batched in one call — much faster and cheaper than one request per chunk). The response is `{"data": [{"embedding": [...]}, ...]}`, in the same order as your input.

`openai/text-embedding-3-small` is a good default: cheap (~$0.02 per million tokens), fast, and produces 1536-dimensional vectors.

Implement `embed_chunks(chunks, model="openai/text-embedding-3-small")`:
1. Build the list of all chunk texts.
2. Make **one** POST request with all of them as a batched `input` list.
3. Walk through `response.json()["data"]` and assign each chunk's `["embedding"]` in order.
4. Return the (mutated) `chunks` list.

In [ ]:
import requests

OPENROUTER_EMBEDDINGS_URL = "https://openrouter.ai/api/v1/embeddings"

def embed_chunks(chunks: list, model: str = "openai/text-embedding-3-small") -> list:
    # TODO:
    # 1. texts = [c["text"] for c in chunks]
    # 2. POST to OPENROUTER_EMBEDDINGS_URL with json={"model": model, "input": texts}
    #    and headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}", "Content-Type": "application/json"}
    # 3. response.json()["data"] is a list of {"embedding": [...]} in the same order as `texts`
    # 4. assign chunk["embedding"] for each chunk, return chunks
    pass

chunks = embed_chunks(chunks)
print(f"Embedding dimension: {len(chunks[0]['embedding'])}")

from tests import rag_service_checks
rag_service_checks.check_exercise_2(chunks)

## Exercise 3: Populate ChromaDB with Real Embeddings

Reuse the exact same `VectorStoreManager` class from the main lab — nothing about ChromaDB or HNSW needs to change; it stores whatever embeddings you give it, real or simulated.

Create a store, add all 25 embedded chunks, and confirm the collection count matches.

In [ ]:
import chromadb
import hashlib

class VectorStoreManager:
    """Manages ChromaDB with HNSW indexing. (Same as the main lab.)"""

    def __init__(self, collection_name="rag_knowledge_base"):
        self.client = chromadb.Client()
        self.collection = self.client.get_or_create_collection(
            name=# TODO:
            metadata= # TODO:
        
        print(f"Collection '{collection_name}' ready (HNSW cosine)")

    def add_documents(self, documents, batch_size=100):
        # TODO:

    def search(self, query_embedding, n_results=5, filter_conditions=None):
        # TODO:

    def get_stats(self):
        # TODO:

# TODO: create a VectorStoreManager, add all 25 embedded `chunks` to it, and store
# its get_stats() dict in a variable called `stats`.
store = None
stats = None

from tests import rag_service_checks
rag_service_checks.check_exercise_3(stats, expected_count=25)

## Exercise 4: Embed a Query via OpenRouter

Retrieval only works if the query and the documents live in the *same* embedding space — so queries must be embedded with the exact same model as the documents.

Implement `embed_query(query, model="openai/text-embedding-3-small")`: same request pattern as Exercise 2, just with a single string as `input` instead of a batch list, returning `response.json()["data"][0]["embedding"]`.

In [ ]:
def embed_query(query: str, model: str = "openai/text-embedding-3-small") -> list:
    # TODO: same POST pattern as embed_chunks, but with a single string as "input"
    # instead of a list. Return response.json()["data"][0]["embedding"].
    pass

demo_query = "How does the retriever avoid making things up?"
emb_a = embed_query(demo_query)
emb_b = embed_query(demo_query)  # embed the same query again
print(f"Query embedding dimension: {len(emb_a)}")

from tests import rag_service_checks
rag_service_checks.check_exercise_4(emb_a, len(chunks[0]["embedding"]), repeat_embedding=emb_b)

## Exercise 5: Semantic Search on Paraphrased Queries

This is the payoff. The main lab's simulated `_embed_query` only boosts the right region of the vector if the query literally contains a trigger word like `"rag"`, `"vector"`, or `"chunk"`. Ask it something that means the same thing but uses different words, and it has no way to know.

A real embedding model doesn't have that limitation. Implement `semantic_search(store, query, n_results=3)`: embed the query with `embed_query`, then call `store.search(...)`. Run it on the two paraphrased test queries below — neither shares much vocabulary with its target chunk's text — and confirm the right chunk shows up near the top.

In [ ]:
def semantic_search(store, query: str, n_results: int = 3) -> list:
    # TODO: embed_query(query), then store.search(query_embedding, n_results=n_results)
    pass

test_queries = {
    # No shared keywords with "hallucination", but the same underlying question
    "How does the retriever avoid making things up?": {"Hallucination in LLMs", "RAG Fundamentals"},
    # No shared keywords with "HNSW", but describes exactly what it is
    "What algorithm organizes vectors into layered graphs for fast search?": {"HNSW Indexing"},
}

results_by_query = {}
for q in test_queries:
    results = semantic_search(store, q, n_results=3)
    results_by_query[q] = results
    print(f"\nQuery: {q}")
    for r in results:
        print(f"  [{r['score']:.3f}] {r['metadata']['title']}")

from tests import rag_service_checks
rag_service_checks.check_exercise_5(results_by_query, test_queries)

## Exercise 6: Score Thresholding on Real Similarity Scores

Implement `filtered_search(store, query, n_results=10, min_score=0.0)`: run a semantic search (reuse `semantic_search` for the unfiltered list), then keep only results with `score >= min_score`.

In [ ]:
def filtered_search(store, query: str, n_results: int = 10, min_score: float = 0.0) -> list:
    # TODO: get up to n_results results for `query`, then keep only those with score >= min_score.
    pass

demo_query2 = "What is RAG?"
loose_results = filtered_search(store, demo_query2, n_results=10, min_score=0.0)
strict_results = filtered_search(store, demo_query2, n_results=10, min_score=0.5)

print(f"Unfiltered: {len(loose_results)} results")
print(f"min_score=0.5: {len(strict_results)} results")
for r in strict_results:
    print(f"  [{r['score']:.3f}] {r['metadata']['title']}")

from tests import rag_service_checks
rag_service_checks.check_exercise_6(loose_results, strict_results, min_score=0.5)

## Reflection

1. **Determinism**: Exercise 4 checked that embedding the same query twice gives (nearly) identical vectors. Why couldn't the main lab's simulated `_embed_query` make that same guarantee?
2. **Paraphrase gap**: In Exercise 5, try running the main lab's *simulated* `_embed_query` on the same two paraphrased queries against the *simulated* store. What do you get back, and why?
3. **Cost & latency**: Every call in this notebook cost a small amount of OpenRouter credit and took network round-trip time; the main lab's simulated embedding was instant and free. At what point in a real project is that tradeoff worth it?
4. **Batching**: Exercise 2 embedded all 25 chunks in one API call instead of 25 separate calls. What would have gone wrong (cost, latency, rate limits) if you'd called `embed_query` in a loop instead?

*Your answers here:*
1. ...
2. ...
3. ...
4. ...